# 5. Outcome dataset: what happened after time zero

**What it does:** one row per patient with treatment, care process and outcomes, all timed from
time zero (never used for clustering):

- **Surgery:** first SDH operation (burr hole, craniotomy, twist drill, membrane excision ...), its
  code, hours from time zero, hospital; repeat operation. Operations *before* time zero (e.g. the
  brain surgery an in-hospital SDH followed) are ignored.
- **Other procedures:** MMA embolization, embolization that is probably aneurysm treatment, EVD,
  ventilation, craniectomy.
- **Care process:** ICU (hours from time zero), ER neurosurgery consult, palliative care code.
- **Pathway:** surgery < 24 h, surgery >= 24 h, MMA embolization only, no surgery with palliative care, no surgery.
- **Outcomes:** in-hospital death, death within 48 h, length of stay (from time zero; survivors only
  as well), discharge category, favourable discharge, 30-day readmission, SDH readmission within
  90 days, days alive and out of hospital to day 30.

**Output:** `patient_level/outcomes.csv`; aggregate check tables.
**Time:** a few minutes.

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Run

In [ ]:
%run ../04_outcomes.py

## Results

In [ ]:
show("check_intervention_codes.csv", rows=60, note="every CCI code in the surgery families and the group it was given");

**Look for (ideally with a neurosurgeon):** each code in the right group; `1JW51` embolizations with
an aneurysm code are labelled *probable aneurysm*; codes with no group that should have one.
`with_time` = how many have a real time (not 00:00).

In [ ]:
show("check_discharge_disposition.csv", note="CIHI discharge codes and the category given");

**Look for:** every code mapped to a sensible category (home, long-term care, died ...).

In [ ]:
show("check_surgery_hospitals.csv", note="SDH operations per hospital; >= 20 = neurosurgical centre");

In [ ]:
o = io.read_patient_level("outcomes.csv")
summary = {}
for c in ["pathway", "first_surgery_type", "discharge_category"]:
    summary[c] = o[c].fillna("(none)").value_counts()
for c, s in summary.items():
    display(s.rename(f"{c}: patients").to_frame().map(io.suppress))
flags = ["cranial_surgery", "mma_embolization", "icu_admission", "er_neurosurgery_consult",
         "palliative_care_coded", "died_in_hospital", "died_within_48h", "favourable_discharge"]
tf = pd.DataFrame({c: [io.suppress(int(o[c].astype(str).str.lower().eq("true").sum()))] for c in flags if c in o},
                  index=["patients"]).T
display(tf)

## Share

In [ ]:
share(["check_intervention_codes.csv", "check_discharge_disposition.csv", "check_surgery_hospitals.csv"])